# Get Embeddings With SToFM

Input: adata (RNA seq, coordinates)

Output: embeddings npy

## Colab pre-requisites

In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# mount to google drive
from google.colab import drive
# drive.flush_and_unmount()
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ST_FM_Benchmark/SToFM

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ST_FM_Benchmark/SToFM


In [2]:
# Verify versions:
# Python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
# Torch: 2.8.0+cu126 CUDA: 12.6
# numpy:  2.0.2
# CXX11_ABI: True

# import sys, torch, numpy as np
# print("Python:", sys.version)
# print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
# print("numpy: ", np.__version__)
# try:
#     print("CXX11_ABI:", torch._C._GLIBCXX_USE_CXX11_ABI)  # 0=FALSE, 1=TRUE
# except Exception as e:
#     print(e)

Python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
Torch: 2.8.0+cu126 CUDA: 12.6
numpy:  2.0.2
CXX11_ABI: True


In [3]:
# install dependencies; see SToFM/requirements.txt
!pip install torch==2.8.0+cu126 -f https://download.pytorch.org/whl/torch_stable.html
!pip install 'rapids-singlecell[rapids12]' --extra-index-url=https://pypi.nvidia.com
!pip install cupy-cuda12x
!pip install transformers
!pip install datasets
!pip install pandas
!pip install scanpy
!pip install scikit-learn
!pip install seaborn
!pip install matplotlib
!pip install scipy
!pip install loompy
!pip3 install igraph

Looking in links: https://download.pytorch.org/whl/torch_stable.html
Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 139.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 110.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.0/565.0 MB 51.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.6/27.6 MB 46.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 41.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.9/135.9 kB 162.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 86.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 GB 26.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━

## Run SToFM to get embeddings

In [6]:
import scanpy as sc

adata = sc.read_h5ad('data/1_visium.h5ad')
print(adata)

AnnData object with n_obs × n_vars = 4221 × 33538
    obs: 'in_tissue', 'array_row', 'array_col', 'Region', 'ground_truth', 'protocol'
    var: 'gene_ids', 'feature_types', 'genome', 'gene_name'
    uns: 'spatial'
    obsm: 'spatial'


In [2]:
# preprocess; for h5ad files in one folder, generate a `hf.dataset`, `ce_emb.npy`, `data.h5ad`.
from preprocessing.preprocess import SToFMTranscriptomeTokenizer
import scanpy as sc
import pickle as pkl

mouseid2humanid = pkl.load(open("preprocessing/mouseid2humanid.pkl", "rb"))
tk = SToFMTranscriptomeTokenizer({}, nproc=4)
adata = sc.read_h5ad(f'data/1_visium.h5ad')

adata.var['ensembl_id'] = [mouseid2humanid[gene_id] if gene_id in mouseid2humanid else gene_id for gene_id in adata.var['gene_ids']]
adata.obs['n_counts'] = adata.X.sum(axis=1)
adata.obs['filter_pass'] = True
tokenized_cells, cell_metadata = tk.tokenize_anndata(adata)
tokenized_dataset = tk.create_dataset(tokenized_cells, cell_metadata)
tokenized_dataset.save_to_disk(f"data/hf.dataset")
adata.write(f"data/data.h5ad")

/content/drive/MyDrive/ST_FM_Benchmark/SToFM/preprocessing/preprocess.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in data.var["ensembl_id"][coding_miRNA_loc]
/content/drive/MyDrive/ST_FM_Benchmark/SToFM/preprocessing/preprocess.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = data.var["ensembl_id"][coding_miRNA_loc]


Map (num_proc=4):   0%|          | 0/4221 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/4221 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4221 [00:00<?, ? examples/s]

In [5]:
# run get embeddings
%%bash

export CUDA_VISIBLE_DEVICES=0
python -u get_embeddings.py \
--cell_encoder_path ckpt/cell_encoder \
--data_path data/ \
--batch_size 4 \
--config_path ckpt/config.json \
--model_path ckpt/se2transformer.pth \
--output_filename ../data/1_visium_stofm.npy \
--seed 1

Encode cell data//data.h5ad
Load data//data.h5ad
Cell 4221, Sub-slice 9


2025-10-05 21:03:57.220058: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-05 21:03:57.239051: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759698237.261882   18676 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759698237.268637   18676 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1759698237.286213   18676 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 